# CARLA Camera Diagnostics Notebook

This notebook checks whether RGB camera output behaves correctly across CARLA versions, with a focus on **non-standard image sizes and aspect ratios**.

learning goals — **spawn a vehicle, attach a camera, view camera output, and compare behavior across versions** — 

- cleaner setup and cleanup
- explicit version check
- several camera presets instead of a single square stream
- a reusable spawn helper
- side-by-side still captures for fast visual comparison

Suggested use:
1. Launch one CARLA version.
2. Run this notebook and inspect the saved captures / preview windows.
3. Repeat with another CARLA version and compare results.


In [7]:
import carla
import cv2
import numpy as np
import time
import random
from pathlib import Path
from IPython.display import display, Image


In [10]:
# Connection and output settings
HOST = "localhost"
PORT = 2000
TIMEOUT = 10.0

OUTPUT_DIR = Path("camera_diagnostics_output")
OUTPUT_DIR.mkdir(exist_ok=True)

client = carla.Client(HOST, PORT)
client.set_timeout(TIMEOUT)

world = client.get_world()
carla_map = world.get_map()

print("Connected to:", client.get_server_version())
print("Map:", carla_map.name)
print("Output folder:", OUTPUT_DIR.resolve())


RuntimeError: time-out of 10000ms while waiting for the simulator, make sure the simulator is ready and connected to localhost:2000

In [ ]:
# Helpers
created_actors = []

def remember(actor):
    created_actors.append(actor)
    return actor

def cleanup():
    for actor in reversed(created_actors):
        try:
            if actor.is_listening:
                actor.stop()
        except Exception:
            pass
        try:
            actor.destroy()
        except Exception:
            pass
    created_actors.clear()
    print("Cleanup complete.")

def choose_vehicle_blueprint(world, preferred_patterns=None):
    library = world.get_blueprint_library()
    preferred_patterns = preferred_patterns or ["vehicle.tesla.model3", "vehicle.audi.tt", "vehicle.mini.cooper_s"]
    for pattern in preferred_patterns:
        matches = library.filter(pattern)
        if matches:
            return random.choice(matches)
    return random.choice(library.filter("vehicle.*"))

def spawn_vehicle(world, spawn_index=0):
    spawn_points = world.get_map().get_spawn_points()
    if not spawn_points:
        raise RuntimeError("No spawn points available on this map.")
    transform = spawn_points[spawn_index % len(spawn_points)]
    bp = choose_vehicle_blueprint(world)
    vehicle = world.try_spawn_actor(bp, transform)
    if vehicle is None:
        raise RuntimeError("Vehicle spawn failed; try a different spawn index.")
    return remember(vehicle), transform

def set_spectator_above(world, target_transform, height=6.0, pitch=-25.0):
    spectator = world.get_spectator()
    view_transform = carla.Transform(
        carla.Location(
            x=target_transform.location.x,
            y=target_transform.location.y,
            z=target_transform.location.z + height,
        ),
        carla.Rotation(pitch=pitch, yaw=target_transform.rotation.yaw, roll=0.0)
    )
    spectator.set_transform(view_transform)


In [ ]:
# Spawn a vehicle and position spectator
cleanup()  # safe rerun

vehicle, vehicle_transform = spawn_vehicle(world, spawn_index=3)
vehicle.set_autopilot(True)
set_spectator_above(world, vehicle_transform)

print("Vehicle:", vehicle.type_id)
print("Spawned at:", vehicle_transform)


In [ ]:
# Camera diagnostic presets
# These intentionally include both standard and non-standard resolutions.
camera_presets = [
    {"name": "square_360", "width": 360, "height": 360, "fov": 90},
    {"name": "wide_640x240", "width": 640, "height": 240, "fov": 100},
    {"name": "tall_300x500", "width": 300, "height": 500, "fov": 80},
]

CAMERA_OFFSET = carla.Transform(
    carla.Location(x=1.2, z=1.5),
    carla.Rotation(pitch=-5.0)
)

for preset in camera_presets:
    print(preset)


In [ ]:
# Camera capture utilities
def build_rgb_camera(world, width, height, fov):
    bp = world.get_blueprint_library().find("sensor.camera.rgb")
    bp.set_attribute("image_size_x", str(width))
    bp.set_attribute("image_size_y", str(height))
    bp.set_attribute("fov", str(fov))
    return bp

def capture_single_frame(world, vehicle, preset, warmup_frames=15):
    width = preset["width"]
    height = preset["height"]
    bp = build_rgb_camera(world, width, height, preset["fov"])
    camera = remember(world.spawn_actor(bp, CAMERA_OFFSET, attach_to=vehicle))

    frame_store = {"image": None}

    def _callback(image):
        array = np.frombuffer(image.raw_data, dtype=np.uint8)
        array = array.reshape((image.height, image.width, 4))[:, :, :3]
        frame_store["image"] = array

    camera.listen(_callback)

    # Wait briefly for frames to arrive.
    for _ in range(warmup_frames):
        if frame_store["image"] is not None:
            break
        time.sleep(0.1)

    if frame_store["image"] is None:
        raise RuntimeError(f"No frame received for preset {preset['name']}")

    image = frame_store["image"].copy()
    camera.stop()
    camera.destroy()
    created_actors.remove(camera)
    return image

def save_bgr_png(rgb_array, output_path):
    bgr = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(output_path), bgr)


In [ ]:
# Capture one still image for each preset
captures = {}

for preset in camera_presets:
    frame = capture_single_frame(world, vehicle, preset)
    captures[preset["name"]] = frame

    out_path = OUTPUT_DIR / f"{preset['name']}.png"
    save_bgr_png(frame, out_path)
    print(f"Saved: {out_path}")


In [ ]:
# Build a simple contact sheet for quick comparison
def resize_to_height(image, target_height):
    h, w = image.shape[:2]
    scale = target_height / h
    new_w = max(1, int(round(w * scale)))
    return cv2.resize(image, (new_w, target_height), interpolation=cv2.INTER_AREA)

def add_label(image_rgb, text):
    canvas = image_rgb.copy()
    cv2.rectangle(canvas, (0, 0), (canvas.shape[1], 32), (0, 0, 0), thickness=-1)
    cv2.putText(canvas, text, (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    return canvas

labeled = []
target_height = 260

for preset in camera_presets:
    img = captures[preset["name"]]
    label = f"{preset['name']} ({preset['width']}x{preset['height']}, fov={preset['fov']})"
    img = resize_to_height(img, target_height)
    img = add_label(img, label)
    labeled.append(img)

sheet = cv2.hconcat([cv2.cvtColor(img, cv2.COLOR_RGB2BGR) for img in labeled])
sheet_path = OUTPUT_DIR / "camera_contact_sheet.png"
cv2.imwrite(str(sheet_path), sheet)

print("Saved contact sheet:", sheet_path)
display(Image(filename=str(sheet_path)))


In [ ]:
# Optional live preview with one preset
LIVE_PRESET = camera_presets[1]  # choose one of the presets above
live_bp = build_rgb_camera(world, LIVE_PRESET["width"], LIVE_PRESET["height"], LIVE_PRESET["fov"])
live_camera = remember(world.spawn_actor(live_bp, CAMERA_OFFSET, attach_to=vehicle))

live_buffer = {"image": np.zeros((LIVE_PRESET["height"], LIVE_PRESET["width"], 3), dtype=np.uint8)}

def live_callback(image):
    array = np.frombuffer(image.raw_data, dtype=np.uint8)
    array = array.reshape((image.height, image.width, 4))[:, :, :3]
    live_buffer["image"] = array

live_camera.listen(live_callback)

print("Press q in the OpenCV window to stop preview.")
while True:
    preview_bgr = cv2.cvtColor(live_buffer["image"], cv2.COLOR_RGB2BGR)
    cv2.imshow("CARLA RGB Diagnostic Preview", preview_bgr)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cv2.destroyAllWindows()
live_camera.stop()
live_camera.destroy()
created_actors.remove(live_camera)


## What to compare across CARLA versions

When you rerun this notebook on another CARLA version, compare:

- whether the live preview looks visually correct
- whether square / wide / tall captures preserve geometry correctly
- whether saved PNG files have the expected dimensions
- whether the contact sheet reveals stretching, scrambling, or channel corruption

This makes version-to-version comparison faster than checking a single camera stream manually.


In [ ]:
# Inspect saved image shapes
for name, img in captures.items():
    print(name, "->", img.shape)


In [ ]:
# Final cleanup
cleanup()
cv2.destroyAllWindows()
